In [1]:
import pandas as pd

In [2]:
# 1. KIỂM TRA CARD ĐỒ HỌA (GPU)
print("========== 1. THÔNG TIN GPU ==========")
!nvidia-smi

import torch
print("\n========== 2. TRẠNG THÁI PYTORCH ==========")
print(f"PyTorch đã nhận GPU chưa?: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Đang sử dụng Card: {torch.cuda.get_device_name(0)}")
    print(f"Dung lượng VRAM: {round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1)} GB")

# 3. KIỂM TRA CPU VÀ RAM HỆ THỐNG
print("\n========== 3. THÔNG TIN CPU & RAM ==========")
print("CPU:")
!lscpu | grep 'Model name'
!lscpu | grep '^CPU(s):'
print("\nRAM Hệ thống:")
!free -h

========== 1. THÔNG TIN GPU ==========
Fri Mar 13 09:49:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!mkdir -p /content/drive/MyDrive/HALO_Train_Data

In [6]:
# 1. Kết nối Colab với Drive của bạn (nó sẽ hiện bảng hỏi quyền, bấm Cho phép)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# Đã thêm HALO_Train_Data vào đường dẫn
!cp /content/drive/MyDrive/HALO_Train_Data/amr_ws.zip /content/
!unzip -q /content/amr_ws.zip -d /content/

In [24]:
%%bash
set -euo pipefail

LOG_FILE=/content/halo_setup_train.log
rm -f "${LOG_FILE}"
exec > >(tee -a "${LOG_FILE}") 2>&1

run_step () {
  STEP_NAME="$1"
  shift
  echo "========== ${STEP_NAME} =========="
  "$@"
}

HALO_DIR=/content/amr_ws/HALO
cd "${HALO_DIR}"

run_step "APT update" sudo apt-get update -y
run_step "Install base deps" sudo apt-get install -y build-essential cmake git libopencv-dev libeigen3-dev python3-dev pkg-config

run_step "Upgrade pip" python3 -m pip install -U pip setuptools wheel
run_step "Install Python deps" python3 -m pip install casadi pybind11 gym==0.26.2 tensorboard tensorboardx pyglet==1.5.15 socialforce
run_step "Install torch" python3 -m pip install torch torchvision
run_step "Install dgl" python3 -m pip install dgl -f https://data.dgl.ai/wheels/cu121/repo.html

cd "${HALO_DIR}/src/ocp_planner"
mkdir -p extern
if [ ! -d extern/pybind11 ]; then
  run_step "Clone pybind11" git clone https://github.com/pybind/pybind11.git extern/pybind11
fi
cp CMakeLists_standalone.txt CMakeLists.txt

PY_EXE=$(which python3)
PY_INC=$(${PY_EXE} - <<'PY'
import sysconfig
print(sysconfig.get_paths()["include"])
PY
)
CASADI_DIR=$(${PY_EXE} - <<'PY'
import casadi, pathlib
print(pathlib.Path(casadi.__file__).resolve().parent)
PY
)
CASADI_LIB=$(${PY_EXE} - <<'PY'
import casadi, pathlib, glob
d = pathlib.Path(casadi.__file__).resolve().parent
cands = sorted(glob.glob(str(d / 'libcasadi.so*')))
print(cands[-1] if cands else '')
PY
)

echo "Using PY_EXE=${PY_EXE}"
echo "Using PY_INC=${PY_INC}"
echo "Using CASADI_DIR=${CASADI_DIR}"
echo "Using CASADI_LIB=${CASADI_LIB}"
if [ -z "${CASADI_LIB}" ]; then
  echo "ERROR: libcasadi.so not found inside pip casadi package"
  exit 1
fi

sed -i "s|set(CASADI_INCLUDE_DIR.*|set(CASADI_INCLUDE_DIR \"${CASADI_DIR}/include\")|g" CMakeLists.txt
sed -i "s|set(CASADI_LIB_DIR.*|set(CASADI_LIB_DIR \"${CASADI_DIR}\")|g" CMakeLists.txt
sed -i "s|set(PYTHON_EXECUTABLE.*|set(PYTHON_EXECUTABLE \"${PY_EXE}\")|g" CMakeLists.txt
sed -i "s|set(PYTHON_INCLUDE_DIRECTORY.*|set(PYTHON_INCLUDE_DIRECTORY \"${PY_INC}\")|g" CMakeLists.txt

build_once () {
  EXTRA_CXX_FLAGS="$1"
  rm -rf build
  mkdir -p build
  cd build
  run_step "CMake configure" cmake .. -DCMAKE_BUILD_TYPE=Release -DCMAKE_CXX_FLAGS="${EXTRA_CXX_FLAGS}"
  run_step "CMake build" make -j2
  run_step "CMake install" make install
  cd ..
}

export LD_LIBRARY_PATH="${CASADI_DIR}:${LD_LIBRARY_PATH:-}"

echo "Try build with default ABI"
build_once ""

cd "${HALO_DIR}/drl_moudle"
if ! python3 -c "import ocp_planner_py; print('ocp_planner_py OK (default ABI)')"; then
  echo "Import failed -> retry with ABI=0 compatibility"
  cd "${HALO_DIR}/src/ocp_planner"
  build_once "-D_GLIBCXX_USE_CXX11_ABI=0"
  cd "${HALO_DIR}/drl_moudle"
  python3 -c "import ocp_planner_py; print('ocp_planner_py OK (ABI=0)')"
fi

run_step "Train PPO" python3 train_ppo.py --output_dir train_data/colab_run --total_timesteps 200000

echo "[OK] Finished. Log saved at ${LOG_FILE}"

========== APT update ==========
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://packages.ros.org/ros2/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jam

CalledProcessError: Command 'b'set -euo pipefail\n\nLOG_FILE=/content/halo_setup_train.log\nrm -f "${LOG_FILE}"\nexec > >(tee -a "${LOG_FILE}") 2>&1\n\nrun_step () {\n  STEP_NAME="$1"\n  shift\n  echo "========== ${STEP_NAME} =========="\n  "$@"\n}\n\nHALO_DIR=/content/amr_ws/HALO\ncd "${HALO_DIR}"\n\nrun_step "APT update" sudo apt-get update -y\nrun_step "Install base deps" sudo apt-get install -y build-essential cmake git libopencv-dev libeigen3-dev python3-dev pkg-config\n\nrun_step "Upgrade pip" python3 -m pip install -U pip setuptools wheel\nrun_step "Install Python deps" python3 -m pip install casadi pybind11 gym==0.26.2 tensorboard tensorboardx pyglet==1.5.15 socialforce\nrun_step "Install torch" python3 -m pip install torch torchvision\nrun_step "Install dgl" python3 -m pip install dgl -f https://data.dgl.ai/wheels/cu121/repo.html\n\ncd "${HALO_DIR}/src/ocp_planner"\nmkdir -p extern\nif [ ! -d extern/pybind11 ]; then\n  run_step "Clone pybind11" git clone https://github.com/pyb

In [23]:
# Cell 8 - Chẩn đoán nhanh lỗi setup/train gần nhất
import os
log_file = '/content/halo_setup_train.log'
print('Log exists:', os.path.exists(log_file))
if os.path.exists(log_file):
    print('\n===== STEP MARKERS =====')
    !grep -n "^==========" /content/halo_setup_train.log || true
    print('\n===== ABI CHECK =====')
    !grep -nE "default ABI|ABI=0|Import failed|ocp_planner_py OK" /content/halo_setup_train.log || true
    print('\n===== LAST 100 LINES =====')
    !tail -n 100 /content/halo_setup_train.log
    print('\n===== KEY ERROR LINES =====')
    !grep -nEi "error|failed|not found|undefined symbol|CMake Error|Traceback" /content/halo_setup_train.log | tail -n 60 || true
else:
    print('Chưa có log. Hãy chạy Cell 7 trước.')

Log exists: True

===== STEP MARKERS =====

===== LAST 80 LINES =====
Using PY_EXE=/usr/bin/python3
Using PY_INC=/usr/include/python3.12
Using CASADI_DIR=/usr/local/lib/python3.12/dist-packages/casadi
\n========== CMake configure ==========
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found OpenCV: /usr (found version "4.5.4")
-- Found Casadi: /usr/local/lib/python3.12/dist-packages/casadi/libcasadi.so
-- pybind11 v3.1.0 
-- Found Python: /usr/local/bin/python (found suitable version "3.12.12", minimum required is "3.8") found comp